# 03b — Common-Footprint Cropping

Notebooks 01–03 process each instrument's **entire strip**. OHRC is a narrow, very-high-res swath; TMC-2 is wider and coarser; the true ground overlap between any two products is usually a small fraction of either image. Matching full strips means the matchers spend almost all their effort on content that has no counterpart in the other image at all.

This notebook crops OHRC / TMC-2 / IIRS down to the **shared ground area** for each pairing, using the real georeferencing Notebook 03 already computed, *before* scale-space and matching run. Everything downstream (04, 05, 06, 07) then operates on these common-area products instead of the full strips — including the final registered output.

**IIRS has no per-pixel GCPs in this download** (Notebook 03 explicitly skips it). This notebook makes a best-effort attempt to read an approximate 4-corner footprint from the IIRS PDS4 label; if that isn't present, it falls back to using IIRS's full frame uncropped (and TMC-2's full frame too, for that pairing only, since the true overlap location inside TMC-2 is then unknown).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, csv, math
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.transform import from_gcps
from rasterio.control import GroundControlPoint
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

DRIVE_ROOT = '/content/drive/MyDrive/26166_DATA'
RAW_ROOT   = f'{DRIVE_ROOT}/raw'
OUT_ROOT   = f'{DRIVE_ROOT}/processed/common_crop'
os.makedirs(OUT_ROOT, exist_ok=True)

# Same spherical Moon CRS used in Notebook 03.
MOON_PROJ4 = '+proj=longlat +a=1737400 +b=1737400 +no_defs'

with open(f'{DRIVE_ROOT}/processed/manifest_radiometric.csv') as f:
    radio_rows = list(csv.DictReader(f))

geocoded_rows = []
geocoded_manifest = f'{DRIVE_ROOT}/processed/manifest_geocoded.csv'
if os.path.exists(geocoded_manifest):
    with open(geocoded_manifest) as f:
        geocoded_rows = list(csv.DictReader(f))

def get_radio_row(instrument):
    match = next((r for r in radio_rows if r['instrument'] == instrument), None)
    assert match, f'No radiometric product for {instrument}'
    return match

def get_geocoded_row(instrument):
    return next((r for r in geocoded_rows if r['instrument'] == instrument), None)

print(f'{len(radio_rows)} radiometric products, {len(geocoded_rows)} geocoded products found.')

## 1. Crop / passthrough helpers

In [ ]:
def bounds_intersection(b1, b2):
    left, right = max(b1.left, b2.left), min(b1.right, b2.right)
    bottom, top = max(b1.bottom, b2.bottom), min(b1.top, b2.top)
    if left >= right or bottom >= top:
        return None
    return left, bottom, right, top

def crop_to_bbox(src_path, bbox, out_path):
    """Crop a georeferenced GeoTIFF to a lon/lat bbox, keeping native resolution
    (no resampling — just a windowed read). Works regardless of the transform's
    row/col orientation (rasterio's Window.from_bounds assumes a specific sign
    convention and can raise 'Bounds and transform are inconsistent' on GCP-
    derived transforms that don't match it — this computes the window directly
    from the inverse transform instead, which is orientation-agnostic).
    Returns out_path, or None if the window collapses to nothing."""
    left, bottom, right, top = bbox
    with rasterio.open(src_path) as src:
        inv = ~src.transform
        corners = [(left, bottom), (left, top), (right, bottom), (right, top)]
        cols, rows = zip(*(inv * (x, y) for x, y in corners))
        window = Window(col_off=min(cols), row_off=min(rows),
                         width=max(cols) - min(cols), height=max(rows) - min(rows))
        window = window.round_offsets().round_lengths()
        window = window.intersection(Window(0, 0, src.width, src.height))
        if window.width < 8 or window.height < 8:
            return None
        arr = src.read(1, window=window)
        new_transform = src.window_transform(window)
        src_crs = src.crs

    # Fresh minimal profile — NOT src.profile.copy(), which can carry over
    # TILED/BLOCKXSIZE/BLOCKYSIZE settings that conflict once rewritten
    # (that's the 'BLOCKXSIZE can only be used with TILED=YES' warning).
    profile = {'driver': 'GTiff', 'height': arr.shape[0], 'width': arr.shape[1],
               'count': 1, 'dtype': str(arr.dtype), 'crs': src_crs, 'transform': new_transform}
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(arr, 1)
    return out_path

def copy_full_frame(src_path, out_path):
    """No geographic info available (or no overlap found) — pass the product
    through unchanged, so the pipeline still has something to work with."""
    with rasterio.open(src_path) as src:
        arr = src.read(1)
        src_crs, src_transform = src.crs, src.transform
    profile = {'driver': 'GTiff', 'height': arr.shape[0], 'width': arr.shape[1],
               'count': 1, 'dtype': str(arr.dtype)}
    if src_crs is not None:
        profile.update({'crs': src_crs, 'transform': src_transform})
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(arr, 1)
    return out_path

## 2. OHRC ↔ TMC-2 — crop to their real, GCP-derived overlap

Both products were georeferenced with real GCPs in Notebook 03, so this is a straightforward geographic intersection.

In [ ]:
common_rows = []

ohrc_geo = get_geocoded_row('ohrc')
tmc2_geo = get_geocoded_row('tmc2')
assert ohrc_geo and tmc2_geo, 'OHRC/TMC-2 must be georeferenced by Notebook 03 first.'

with rasterio.open(ohrc_geo['geocoded_path']) as s: ohrc_bounds = s.bounds
with rasterio.open(tmc2_geo['geocoded_path']) as s: tmc2_bounds = s.bounds

bbox_ohrc_tmc2 = bounds_intersection(ohrc_bounds, tmc2_bounds)

if bbox_ohrc_tmc2 is None:
    print('OHRC and TMC-2 footprints do NOT overlap at all — check the footprint plot')
    print('from Notebook 03 §4 before going any further with this pair.')
else:
    out_dir = f'{OUT_ROOT}/ohrc_to_tmc2'
    os.makedirs(out_dir, exist_ok=True)
    ohrc_common = crop_to_bbox(ohrc_geo['geocoded_path'], bbox_ohrc_tmc2, f'{out_dir}/ohrc_common.tif')
    tmc2_common = crop_to_bbox(tmc2_geo['geocoded_path'], bbox_ohrc_tmc2, f'{out_dir}/tmc2_common.tif')
    print(f'ohrc_to_tmc2: cropped to shared bbox {bbox_ohrc_tmc2}')
    with rasterio.open(ohrc_common) as s: print('  ohrc common size:', s.width, 'x', s.height)
    with rasterio.open(tmc2_common) as s: print('  tmc2 common size:', s.width, 'x', s.height)

    note = 'cropped to real ohrc∩tmc2 GCP-derived bbox'
    common_rows.append({'pair': 'ohrc_to_tmc2', 'role': 'fine', 'instrument': 'ohrc',
                         'common_path': ohrc_common, 'is_cropped': True, 'note': note})
    common_rows.append({'pair': 'ohrc_to_tmc2', 'role': 'coarse', 'instrument': 'tmc2',
                         'common_path': tmc2_common, 'is_cropped': True, 'note': note})

## 3. TMC-2 ↔ IIRS — best-effort approximate footprint, with a safe fallback

IIRS has no per-pixel GCPs here, so we try to read a 4-corner `Bounding_Coordinates` block straight out of its PDS4 label (namespace-agnostic search, since different PDS4 discipline dictionaries name this differently). If found, those 4 corners are used as GCPs (like Notebook 03 does with real GCPs) to build an **approximate** affine transform — good enough to select a common crop region, not a substitute for the real per-pixel georeferencing OHRC/TMC-2 have.

If no such block exists, we fall back to using **IIRS's full frame** uncropped — and TMC-2's full frame too for this pairing, since without any IIRS footprint we have no way of knowing which part of TMC-2 to keep.

In [ ]:
def find_iirs_label(product_stem):
    root = Path(RAW_ROOT) / 'iirs'
    for xml_path in root.rglob('*.xml'):
        if xml_path.stem == product_stem:
            return xml_path
    return None

def try_parse_bounding_corners(xml_path):
    """Best-effort, namespace-agnostic search for west/east/north/south
    bounding-coordinate tags anywhere in the label. Returns (west, east, north,
    south) in degrees, or None if not present / not parseable."""
    try:
        tree = ET.parse(xml_path)
    except Exception:
        return None
    tags = {}
    for el in tree.getroot().iter():
        local = el.tag.split('}')[-1].lower()
        if 'boundingcoordinate' in local and el.text:
            tags[local] = el.text
    needed = ['westboundingcoordinate', 'eastboundingcoordinate',
              'northboundingcoordinate', 'southboundingcoordinate']
    if not all(k in tags for k in needed):
        return None
    try:
        w, e = float(tags['westboundingcoordinate']), float(tags['eastboundingcoordinate'])
        n, s = float(tags['northboundingcoordinate']), float(tags['southboundingcoordinate'])
        return w, e, n, s
    except ValueError:
        return None

In [ ]:
iirs_row = get_radio_row('iirs')
iirs_label = find_iirs_label(iirs_row['product'])
iirs_corners = try_parse_bounding_corners(iirs_label) if iirs_label else None

out_dir = f'{OUT_ROOT}/tmc2_to_iirs'
os.makedirs(out_dir, exist_ok=True)
# NOTE: pairing order matches 04's pairings list — fine='tmc2' (finer GSD,
# gets pyramid-decimated), coarse='iirs' (coarsest GSD, stays native).

if iirs_corners:
    w, e, n, s = iirs_corners
    print(f'IIRS label has bounding coordinates: W={w} E={e} N={n} S={s} — using them.')

    with rasterio.open(iirs_row['radiometric_path']) as src:
        h_px, w_px = src.height, src.width
        arr = src.read(1)

    gcps = [
        GroundControlPoint(row=0,    col=0,    x=w, y=n),
        GroundControlPoint(row=0,    col=w_px, x=e, y=n),
        GroundControlPoint(row=h_px, col=0,    x=w, y=s),
        GroundControlPoint(row=h_px, col=w_px, x=e, y=s),
    ]
    iirs_transform = from_gcps(gcps)
    iirs_geo_tmp = f'{out_dir}/iirs_approx_geo.tif'
    # Fresh minimal profile (see crop_to_bbox note above re: TILED/BLOCKXSIZE).
    profile = {'driver': 'GTiff', 'height': h_px, 'width': w_px, 'count': 1,
               'dtype': str(arr.dtype), 'crs': MOON_PROJ4, 'transform': iirs_transform}
    with rasterio.open(iirs_geo_tmp, 'w', **profile) as dst:
        dst.write(arr, 1)

    with rasterio.open(iirs_geo_tmp) as s: iirs_bounds = s.bounds
    bbox_tmc2_iirs = bounds_intersection(tmc2_bounds, iirs_bounds)

    if bbox_tmc2_iirs is None:
        print('  Approximate IIRS footprint does NOT overlap TMC-2 — falling back to full frames.')
        iirs_corners = None
    else:
        tmc2_common_i = crop_to_bbox(tmc2_geo['geocoded_path'], bbox_tmc2_iirs, f'{out_dir}/tmc2_common.tif')
        iirs_common   = crop_to_bbox(iirs_geo_tmp, bbox_tmc2_iirs, f'{out_dir}/iirs_common.tif')
        print(f'  cropped to shared bbox {bbox_tmc2_iirs}')
        note = 'cropped to tmc2∩iirs bbox (IIRS georeferenced from label corners — approximate)'
        common_rows.append({'pair': 'tmc2_to_iirs', 'role': 'fine', 'instrument': 'tmc2',
                             'common_path': tmc2_common_i, 'is_cropped': True, 'note': note})
        common_rows.append({'pair': 'tmc2_to_iirs', 'role': 'coarse', 'instrument': 'iirs',
                             'common_path': iirs_common, 'is_cropped': True, 'note': note})

if not iirs_corners:
    print("No usable IIRS footprint found — using IIRS's full frame as an approximate crop "
          "(per your choice). TMC-2 also stays full-frame for this pairing, since its true "
          "overlap with IIRS is unknown.")
    tmc2_full = copy_full_frame(tmc2_geo['geocoded_path'], f'{out_dir}/tmc2_full.tif')
    iirs_full = copy_full_frame(iirs_row['radiometric_path'], f'{out_dir}/iirs_full.tif')
    note = 'NOT geographically cropped — IIRS footprint unknown, full frame used as approximation'
    common_rows.append({'pair': 'tmc2_to_iirs', 'role': 'fine', 'instrument': 'tmc2',
                         'common_path': tmc2_full, 'is_cropped': False, 'note': note})
    common_rows.append({'pair': 'tmc2_to_iirs', 'role': 'coarse', 'instrument': 'iirs',
                         'common_path': iirs_full, 'is_cropped': False, 'note': note})

## 4. Visual check — cropped footprints

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
colors = {'ohrc': 'red', 'tmc2': 'blue', 'iirs': 'green'}
plotted_any = False
for r in common_rows:
    try:
        with rasterio.open(r['common_path']) as src:
            if src.crs is None:
                continue
            b = src.bounds
        ax.add_patch(Rectangle((b.left, b.bottom), b.right - b.left, b.top - b.bottom,
                                fill=False, edgecolor=colors[r['instrument']], linewidth=2,
                                label=f"{r['instrument']} ({r['pair']})"))
        plotted_any = True
    except Exception:
        pass
if plotted_any:
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.autoscale(); ax.set_aspect('equal')
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys())
    plt.title('Common-crop footprints actually used for matching')
    plt.grid(True, alpha=0.3)
plt.show()

for r in common_rows:
    with rasterio.open(r['common_path']) as src:
        print(f"{r['pair']:14s} {r['role']:6s} {r['instrument']:5s} "
              f"{src.width:5d} x {src.height:5d}  cropped={r['is_cropped']}")

## 5. Save the manifest consumed by Notebook 04

In [ ]:
out_manifest = f'{DRIVE_ROOT}/processed/manifest_common_crop.csv'
with open(out_manifest, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['pair', 'role', 'instrument', 'common_path', 'is_cropped', 'note'])
    writer.writeheader()
    writer.writerows(common_rows)
print(f'Saved -> {out_manifest} ({len(common_rows)} rows)')
print()
print('Continue to 04_scale_space.ipynb (updated to read this manifest instead of the full strips).')

---